# Lightweight ML & TinyML Classifiers: Decision Tree, Random Forest & TFLite for Silicon Labs EFR32 (Multi-Target Training)

This notebook trains, evaluates, and exports **Decision Tree**, **Random Forest**, and **TensorFlow Lite (TFLite)** models separately for all 3 critical intraoperative clinical targets:
1. **`Future_Hypotension`** (Mean Arterial Pressure < 65 mmHg)
2. **`Future_Hypoxia`** (Peripheral Oxygen Saturation $\text{SpO}_2 < 90\%$)
3. **`Future_Tachycardia`** (Heart Rate > 100 bpm)

### Enhanced Pipeline Highlights:
- **Expanded Clinical Feature Space**: Integrates base vital sign monitors and newly engineered hemodynamic biomarkers:
  - **Base Vitals**: `Solar8000/HR`, `Solar8000/ART_SBP`, `Solar8000/ART_DBP`, `Solar8000/ART_MBP`, `Solar8000/PLETH_SPO2`, `Solar8000/RR_CO2`, `Solar8000/ETCO2`, `Primus/FIO2`, `Solar8000/BT`
  - **Engineered Biomarkers**: `Feature_Pulse_Pressure` ($SBP - DBP$), `Feature_Shock_Index` ($HR/SBP$), `Feature_Modified_Shock_Index` ($HR/MBP$), `Feature_Rate_Pressure_Product` ($HR \times SBP / 100$), `Feature_HR_Mean_60s`, `Feature_HR_Std_60s`, `Feature_HR_Delta_60s`, `Feature_MBP_Mean_60s`, `Feature_MBP_Std_60s`, `Feature_MBP_Delta_60s`
- **StandardScaler Parameter Persistence (JSON)**: Automatically exports mean, variance, and standard deviation per feature column formatted as:
  `"mean": { feature_name: value }`, `"variance": { feature_name: value }`, `"std": { feature_name: value }`
- **Detailed Training & Evaluation Logs for Decision Trees**: Prints detailed training logs, hyperparameter configurations, tree complexity (depth, node count, leaf count, estimated flash/RAM usage), threshold optimization ($\tau^*$), and full test metric tables.
- **Silicon Labs EFR32 Embedded Deployment**:
  1. **Direct C Header Code**: Exports zero-overhead decision tree C logic (`efr32_decision_tree.h`) requiring ~2 KB Flash and 0 KB dynamic RAM.
  2. **TensorFlow Lite FlatBuffer (`.tflite`)**: Exports `.tflite` model files and C byte array headers (`efr32_model_tflite.h`) ready for Gecko SDK / TFLM (TensorFlow Lite for Microcontrollers).

In [4]:
import os
import gc
import glob
import json
import random
import struct
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    classification_report,
    ConfusionMatrixDisplay
)
import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Scikit-Learn & ML Environment Initialized Successfully.")

Scikit-Learn & ML Environment Initialized Successfully.


In [5]:
# ======================================================
# Load Patient Files (Relative Paths) & Feature Extraction
# ======================================================

# Dynamically resolve project directory and labeled data path
base_dir = os.getcwd()
if os.path.exists(os.path.join(base_dir, "patient_labeled_data")):
    input_dir = os.path.join(base_dir, "patient_labeled_data")
elif os.path.exists(os.path.join(base_dir, "..", "patient_labeled_data")):
    input_dir = os.path.join(base_dir, "..", "patient_labeled_data")
else:
    input_dir = os.path.join("..", "patient_labeled_data")

csv_files = sorted(glob.glob(os.path.join(input_dir, "patient_*_1hz.csv")))

# Complete feature spectrum: Base 9 vitals + Newly engineered hemodynamic features
base_features = [
    "Solar8000/HR",
    "Solar8000/ART_SBP",
    "Solar8000/ART_DBP",
    "Solar8000/ART_MBP",
    "Solar8000/PLETH_SPO2",
    "Solar8000/RR_CO2",
    "Solar8000/ETCO2",
    "Primus/FIO2",
    "Solar8000/BT"
]

engineered_features = [
    "Feature_Pulse_Pressure",
    "Feature_Shock_Index",
    "Feature_Modified_Shock_Index",
    "Feature_Rate_Pressure_Product",
    "Feature_HR_Mean_60s",
    "Feature_HR_Std_60s",
    "Feature_HR_Delta_60s",
    "Feature_MBP_Mean_60s",
    "Feature_MBP_Std_60s",
    "Feature_MBP_Delta_60s"
]

features = base_features + engineered_features

train_val_files, test_files = train_test_split(csv_files, test_size=0.20, random_state=42, shuffle=True)
train_files, val_files = train_test_split(train_val_files, test_size=0.125, random_state=42, shuffle=True)

print(f"Dataset Path : {input_dir}")
print(f"Total Patients: {len(csv_files)} | Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")
print(f"Feature Count : {len(features)} ({len(base_features)} base + {len(engineered_features)} engineered)")

def extract_window_features(arr_window):
    """Extract summary statistical metrics over sliding window."""
    means = np.mean(arr_window, axis=0)
    stds = np.std(arr_window, axis=0)
    mins = np.min(arr_window, axis=0)
    maxs = np.max(arr_window, axis=0)
    slopes = (arr_window[-1] - arr_window[0]) / (len(arr_window) + 1e-5)
    return np.hstack([means, stds, mins, maxs, slopes]).astype(np.float32)

clean_feature_names = [f.split("/")[-1] for f in features]
window_feature_names = [f"{feat}_{stat}" for stat in ["mean", "std", "min", "max", "slope"] for feat in clean_feature_names]

WINDOW_SIZE = 600
STRIDE = 5

def build_dataset_matrix(file_list, target_col, selected_features=features, oversample_factor=3):
    """Build feature matrix X and label vector y from list of patient CSV files."""
    X_list, y_list = [], []
    for file in file_list:
        try:
            df = pd.read_csv(file)
            avail_features = [c for c in selected_features if c in df.columns]
            if len(avail_features) < len(base_features) or target_col not in df.columns:
                continue
            
            df_sub = df[avail_features + [target_col]].ffill().bfill().fillna(0)
            arr = df_sub[avail_features].values.astype(np.float32)
            y_vals = df_sub[target_col].values.astype(np.float32)
            
            for i in range(0, len(arr) - WINDOW_SIZE, STRIDE):
                feat_vec = extract_window_features(arr[i:i + WINDOW_SIZE])
                label = y_vals[i + WINDOW_SIZE]
                
                if np.isnan(label):
                    continue
                    
                X_list.append(feat_vec)
                y_list.append(label)
                
                if label == 1.0 and oversample_factor > 1:
                    for _ in range(oversample_factor - 1):
                        X_list.append(feat_vec)
                        y_list.append(label)
        except Exception:
            continue
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

def save_scaler_params_to_json(scaler, feature_names_list, json_path):
    """
    Save StandardScaler mean, variance, and standard deviation per feature to JSON.
    Format:
    {
        "mean": { feature_name: value },
        "variance": { feature_name: value },
        "std": { feature_name: value }
    }
    """
    scaler_dict = {
        "mean": {feat: float(scaler.mean_[i]) for i, feat in enumerate(feature_names_list)},
        "variance": {feat: float(scaler.var_[i]) for i, feat in enumerate(feature_names_list)},
        "std": {feat: float(scaler.scale_[i]) for i, feat in enumerate(feature_names_list)}
    }
    with open(json_path, "w") as f:
        json.dump(scaler_dict, f, indent=4)
    print(f"[Scaler Export] Saved Scaler parameters to JSON: {json_path}")
    return scaler_dict

def get_optimal_tau(y_t, y_p):
    best_tau, best_f1 = 0.5, -1.0
    for tau in np.linspace(0.01, 0.99, 99):
        f1 = f1_score(y_t, (y_p >= tau).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_tau = f1, tau
    return best_tau

def compute_metrics(y_t, y_p, tau):
    y_b = (y_p >= tau).astype(int)
    auroc = roc_auc_score(y_t, y_p)
    auprc = average_precision_score(y_t, y_p)
    acc = accuracy_score(y_t, y_b)
    bal_acc = balanced_accuracy_score(y_t, y_b)
    prec = precision_score(y_t, y_b, zero_division=0)
    rec = recall_score(y_t, y_b, zero_division=0)
    f1 = f1_score(y_t, y_b, zero_division=0)
    mcc = matthews_corrcoef(y_t, y_b)
    tn, fp, fn, tp = confusion_matrix(y_t, y_b).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return dict(auroc=auroc, auprc=auprc, acc=acc, bal_acc=bal_acc, prec=prec, rec=rec, spec=spec, f1=f1, mcc=mcc, tn=tn, fp=fp, fn=fn, tp=tp)

def print_training_logs(model_name, target_name, model, X_tr, y_tr, X_va, y_va, X_te, y_te, optimal_tau, feature_names_list):
    """Print detailed training, validation, testing, and MCU footprint logs."""
    print("=" * 78)
    print(f" [TRAINING LOGS] {model_name.upper()} | TARGET: {target_name}")
    print("=" * 78)
    print(f"Dataset Dimensions:")
    print(f"  - Training Samples   : {X_tr.shape[0]:,} (Pos: {int(np.sum(y_tr)):,} | Neg: {int(len(y_tr) - np.sum(y_tr)):,})")
    print(f"  - Validation Samples : {X_va.shape[0]:,} (Pos: {int(np.sum(y_va)):,} | Neg: {int(len(y_va) - np.sum(y_va)):,})")
    print(f"  - Testing Samples    : {X_te.shape[0]:,} (Pos: {int(np.sum(y_te)):,} | Neg: {int(len(y_te) - np.sum(y_te)):,})")
    print(f"  - Input Features     : {X_tr.shape[1]}")
    print("-" * 78)
    
    if isinstance(model, DecisionTreeClassifier):
        tree_ = model.tree_
        flash_est_kb = (tree_.node_count * 16) / 1024.0
        print(f"Decision Tree Model Complexity & MCU Profiling:")
        print(f"  - Max Depth Configured : {model.max_depth}")
        print(f"  - Actual Tree Depth    : {tree_.max_depth}")
        print(f"  - Total Nodes Count    : {tree_.node_count}")
        print(f"  - Leaf Nodes Count     : {tree_.n_leaves}")
        print(f"  - Estimated Flash Size : ~{flash_est_kb:.2f} KB (Well within EFR32 512KB/1024KB limit)")
        print(f"  - Dynamic RAM Usage    : 0 KB (Stack-based recursive evaluation)")
        print("-" * 78)
    
    test_probs = model.predict_proba(X_te)[:, 1]
    m_def = compute_metrics(y_te, test_probs, 0.50)
    m_opt = compute_metrics(y_te, test_probs, optimal_tau)
    
    print(f"Optimal Threshold (tau*) on Validation Set : {optimal_tau:.4f}")
    print("-" * 78)
    print(f"Metric                 Default (tau=0.50)       OPTIMAL (tau*={optimal_tau:.2f})")
    print("-" * 78)
    print(f"AUROC (ROC AUC)         : {m_def['auroc']:.4f}                  {m_opt['auroc']:.4f}")
    print(f"AUPRC (PR AUC)          : {m_def['auprc']:.4f}                  {m_opt['auprc']:.4f}")
    print(f"Accuracy                : {m_def['acc']:.4f}                  {m_opt['acc']:.4f}")
    print(f"Balanced Accuracy       : {m_def['bal_acc']:.4f}                  {m_opt['bal_acc']:.4f}")
    print(f"Sensitivity / Recall    : {m_def['rec']:.4f}                  {m_opt['rec']:.4f}")
    print(f"Specificity (TNR)       : {m_def['spec']:.4f}                  {m_opt['spec']:.4f}")
    print(f"Precision (PPV)         : {m_def['prec']:.4f}                  {m_opt['prec']:.4f}")
    print(f"F1 Score                : {m_def['f1']:.4f}                  {m_opt['f1']:.4f}")
    print(f"MCC                     : {m_def['mcc']:.4f}                  {m_opt['mcc']:.4f}")
    print("-" * 78)
    print(f"Confusion Matrix (0.50) : TN={m_def['tn']}, FP={m_def['fp']}, FN={m_def['fn']}, TP={m_def['tp']}")
    print(f"Confusion Matrix (tau*) : TN={m_opt['tn']}, FP={m_opt['fp']}, FN={m_opt['fn']}, TP={m_opt['tp']}")
    print("=" * 78 + "\n")

Dataset Path : /home/logan78/Desktop/SiLabs/patient_labeled_data
Total Patients: 3764 | Train: 2634 | Val: 377 | Test: 753
Feature Count : 19 (9 base + 10 engineered)


In [ ]:
# ===========================================================================
# 1. RANDOM FOREST & DECISION TREE TRAINING: Future_Hypotension
# ===========================================================================

target_Future_Hypotension = "Future_Hypotension"

X_tr, y_tr = build_dataset_matrix(train_files[:300], target_Future_Hypotension, oversample_factor=3)
X_va, y_va = build_dataset_matrix(val_files[:50], target_Future_Hypotension, oversample_factor=1)
X_te, y_te = build_dataset_matrix(test_files[:100], target_Future_Hypotension, oversample_factor=1)

num_feats = X_tr.shape[1]
curr_feature_names = window_feature_names[:num_feats]

scaler_Future_Hypotension = StandardScaler()
X_tr_sc = scaler_Future_Hypotension.fit_transform(X_tr)
X_va_sc = scaler_Future_Hypotension.transform(X_va)
X_te_sc = scaler_Future_Hypotension.transform(X_te)

# 1. Save StandardScaler mean, variance, and std to JSON
scaler_json_path = os.path.join(".", f"scaler_{target_Future_Hypotension}.json")
save_scaler_params_to_json(scaler_Future_Hypotension, curr_feature_names, scaler_json_path)

# 2. Train & Log Decision Tree
print("[Training] Fitting Decision Tree Classifier for Future_Hypotension...")
dt_Future_Hypotension = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dt_Future_Hypotension.fit(X_tr_sc, y_tr)

val_probs_dt = dt_Future_Hypotension.predict_proba(X_va_sc)[:, 1]
optimal_tau_dt_hypo = get_optimal_tau(y_va, val_probs_dt)
print_training_logs("Decision Tree", target_Future_Hypotension, dt_Future_Hypotension, X_tr_sc, y_tr, X_va_sc, y_va, X_te_sc, y_te, optimal_tau_dt_hypo, curr_feature_names)

# Save Decision Tree Model
dt_model_path = os.path.join(".", f"efr32_dt_{target_Future_Hypotension}.joblib")
joblib.dump(dt_Future_Hypotension, dt_model_path)

# 3. Train & Log Random Forest
print("[Training] Fitting Random Forest Classifier for Future_Hypotension...")
clf_Future_Hypotension = RandomForestClassifier(n_estimators=30, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
clf_Future_Hypotension.fit(X_tr_sc, y_tr)

val_probs_rf = clf_Future_Hypotension.predict_proba(X_va_sc)[:, 1]
optimal_tau_rf_hypo = get_optimal_tau(y_va, val_probs_rf)
print_training_logs("Random Forest", target_Future_Hypotension, clf_Future_Hypotension, X_tr_sc, y_tr, X_va_sc, y_va, X_te_sc, y_te, optimal_tau_rf_hypo, curr_feature_names)

rf_model_path = os.path.join(".", f"efr32_rf_{target_Future_Hypotension}.joblib")
joblib.dump(clf_Future_Hypotension, rf_model_path)

# Plot Diagnostic Curves
test_probs_dt = dt_Future_Hypotension.predict_proba(X_te_sc)[:, 1]
test_probs_rf = clf_Future_Hypotension.predict_proba(X_te_sc)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fpr_rf, tpr_rf, _ = roc_curve(y_te, test_probs_rf)
fpr_dt, tpr_dt, _ = roc_curve(y_te, test_probs_dt)
axes[0].plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {roc_auc_score(y_te, test_probs_rf):.3f})", color="darkorange", lw=2)
axes[0].plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC = {roc_auc_score(y_te, test_probs_dt):.3f})", color="navy", linestyle="--", lw=2)
axes[0].plot([0, 1], [0, 1], color="gray", linestyle=":")
axes[0].set_title(f"ROC Curves: {target_Future_Hypotension}")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

cm_dt = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_te, (test_probs_dt >= optimal_tau_dt_hypo).astype(int)), display_labels=["Neg", "Pos"])
cm_dt.plot(ax=axes[1], cmap="Blues", colorbar=False)
axes[1].set_title(f"Decision Tree CM (tau*={optimal_tau_dt_hypo:.2f})")

cm_rf = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_te, (test_probs_rf >= optimal_tau_rf_hypo).astype(int)), display_labels=["Neg", "Pos"])
cm_rf.plot(ax=axes[2], cmap="Greens", colorbar=False)
axes[2].set_title(f"Random Forest CM (tau*={optimal_tau_rf_hypo:.2f})")

plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================================
# 2. RANDOM FOREST & DECISION TREE TRAINING: Future_Hypoxia
# ===========================================================================

target_Future_Hypoxia = "Future_Hypoxia"

X_tr, y_tr = build_dataset_matrix(train_files[:300], target_Future_Hypoxia, oversample_factor=4)
X_va, y_va = build_dataset_matrix(val_files[:50], target_Future_Hypoxia, oversample_factor=1)
X_te, y_te = build_dataset_matrix(test_files[:100], target_Future_Hypoxia, oversample_factor=1)

num_feats = X_tr.shape[1]
curr_feature_names = window_feature_names[:num_feats]

scaler_Future_Hypoxia = StandardScaler()
X_tr_sc = scaler_Future_Hypoxia.fit_transform(X_tr)
X_va_sc = scaler_Future_Hypoxia.transform(X_va)
X_te_sc = scaler_Future_Hypoxia.transform(X_te)

# 1. Save StandardScaler mean, variance, and std to JSON
scaler_json_path = os.path.join(".", f"scaler_{target_Future_Hypoxia}.json")
save_scaler_params_to_json(scaler_Future_Hypoxia, curr_feature_names, scaler_json_path)

# 2. Train & Log Decision Tree
print("[Training] Fitting Decision Tree Classifier for Future_Hypoxia...")
dt_Future_Hypoxia = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dt_Future_Hypoxia.fit(X_tr_sc, y_tr)

val_probs_dt = dt_Future_Hypoxia.predict_proba(X_va_sc)[:, 1]
optimal_tau_dt_hypox = get_optimal_tau(y_va, val_probs_dt)
print_training_logs("Decision Tree", target_Future_Hypoxia, dt_Future_Hypoxia, X_tr_sc, y_tr, X_va_sc, y_va, X_te_sc, y_te, optimal_tau_dt_hypox, curr_feature_names)

# Save Decision Tree Model
dt_model_path = os.path.join(".", f"efr32_dt_{target_Future_Hypoxia}.joblib")
joblib.dump(dt_Future_Hypoxia, dt_model_path)

# 3. Train & Log Random Forest
print("[Training] Fitting Random Forest Classifier for Future_Hypoxia...")
clf_Future_Hypoxia = RandomForestClassifier(n_estimators=30, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
clf_Future_Hypoxia.fit(X_tr_sc, y_tr)

val_probs_rf = clf_Future_Hypoxia.predict_proba(X_va_sc)[:, 1]
optimal_tau_rf_hypox = get_optimal_tau(y_va, val_probs_rf)
print_training_logs("Random Forest", target_Future_Hypoxia, clf_Future_Hypoxia, X_tr_sc, y_tr, X_va_sc, y_va, X_te_sc, y_te, optimal_tau_rf_hypox, curr_feature_names)

rf_model_path = os.path.join(".", f"efr32_rf_{target_Future_Hypoxia}.joblib")
joblib.dump(clf_Future_Hypoxia, rf_model_path)

# Plot Diagnostic Curves
test_probs_dt = dt_Future_Hypoxia.predict_proba(X_te_sc)[:, 1]
test_probs_rf = clf_Future_Hypoxia.predict_proba(X_te_sc)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fpr_rf, tpr_rf, _ = roc_curve(y_te, test_probs_rf)
fpr_dt, tpr_dt, _ = roc_curve(y_te, test_probs_dt)
axes[0].plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {roc_auc_score(y_te, test_probs_rf):.3f})", color="darkorange", lw=2)
axes[0].plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC = {roc_auc_score(y_te, test_probs_dt):.3f})", color="navy", linestyle="--", lw=2)
axes[0].plot([0, 1], [0, 1], color="gray", linestyle=":")
axes[0].set_title(f"ROC Curves: {target_Future_Hypoxia}")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

cm_dt = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_te, (test_probs_dt >= optimal_tau_dt_hypox).astype(int)), display_labels=["Neg", "Pos"])
cm_dt.plot(ax=axes[1], cmap="Blues", colorbar=False)
axes[1].set_title(f"Decision Tree CM (tau*={optimal_tau_dt_hypox:.2f})")

cm_rf = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_te, (test_probs_rf >= optimal_tau_rf_hypox).astype(int)), display_labels=["Neg", "Pos"])
cm_rf.plot(ax=axes[2], cmap="Greens", colorbar=False)
axes[2].set_title(f"Random Forest CM (tau*={optimal_tau_rf_hypox:.2f})")

plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================================
# 3. RANDOM FOREST & DECISION TREE TRAINING: Future_Tachycardia
# ===========================================================================

target_Future_Tachycardia = "Future_Tachycardia"

X_tr, y_tr = build_dataset_matrix(train_files[:300], target_Future_Tachycardia, oversample_factor=3)
X_va, y_va = build_dataset_matrix(val_files[:50], target_Future_Tachycardia, oversample_factor=1)
X_te, y_te = build_dataset_matrix(test_files[:100], target_Future_Tachycardia, oversample_factor=1)

num_feats = X_tr.shape[1]
curr_feature_names = window_feature_names[:num_feats]

scaler_Future_Tachycardia = StandardScaler()
X_tr_sc = scaler_Future_Tachycardia.fit_transform(X_tr)
X_va_sc = scaler_Future_Tachycardia.transform(X_va)
X_te_sc = scaler_Future_Tachycardia.transform(X_te)

# 1. Save StandardScaler mean, variance, and std to JSON
scaler_json_path = os.path.join(".", f"scaler_{target_Future_Tachycardia}.json")
save_scaler_params_to_json(scaler_Future_Tachycardia, curr_feature_names, scaler_json_path)

# 2. Train & Log Decision Tree
print("[Training] Fitting Decision Tree Classifier for Future_Tachycardia...")
dt_Future_Tachycardia = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dt_Future_Tachycardia.fit(X_tr_sc, y_tr)

val_probs_dt = dt_Future_Tachycardia.predict_proba(X_va_sc)[:, 1]
optimal_tau_dt_tachy = get_optimal_tau(y_va, val_probs_dt)
print_training_logs("Decision Tree", target_Future_Tachycardia, dt_Future_Tachycardia, X_tr_sc, y_tr, X_va_sc, y_va, X_te_sc, y_te, optimal_tau_dt_tachy, curr_feature_names)

# Save Decision Tree Model
dt_model_path = os.path.join(".", f"efr32_dt_{target_Future_Tachycardia}.joblib")
joblib.dump(dt_Future_Tachycardia, dt_model_path)

# 3. Train & Log Random Forest
print("[Training] Fitting Random Forest Classifier for Future_Tachycardia...")
clf_Future_Tachycardia = RandomForestClassifier(n_estimators=30, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
clf_Future_Tachycardia.fit(X_tr_sc, y_tr)

val_probs_rf = clf_Future_Tachycardia.predict_proba(X_va_sc)[:, 1]
optimal_tau_rf_tachy = get_optimal_tau(y_va, val_probs_rf)
print_training_logs("Random Forest", target_Future_Tachycardia, clf_Future_Tachycardia, X_tr_sc, y_tr, X_va_sc, y_va, X_te_sc, y_te, optimal_tau_rf_tachy, curr_feature_names)

rf_model_path = os.path.join(".", f"efr32_rf_{target_Future_Tachycardia}.joblib")
joblib.dump(clf_Future_Tachycardia, rf_model_path)

# Plot Diagnostic Curves
test_probs_dt = dt_Future_Tachycardia.predict_proba(X_te_sc)[:, 1]
test_probs_rf = clf_Future_Tachycardia.predict_proba(X_te_sc)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fpr_rf, tpr_rf, _ = roc_curve(y_te, test_probs_rf)
fpr_dt, tpr_dt, _ = roc_curve(y_te, test_probs_dt)
axes[0].plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {roc_auc_score(y_te, test_probs_rf):.3f})", color="darkorange", lw=2)
axes[0].plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC = {roc_auc_score(y_te, test_probs_dt):.3f})", color="navy", linestyle="--", lw=2)
axes[0].plot([0, 1], [0, 1], color="gray", linestyle=":")
axes[0].set_title(f"ROC Curves: {target_Future_Tachycardia}")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

cm_dt = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_te, (test_probs_dt >= optimal_tau_dt_tachy).astype(int)), display_labels=["Neg", "Pos"])
cm_dt.plot(ax=axes[1], cmap="Blues", colorbar=False)
axes[1].set_title(f"Decision Tree CM (tau*={optimal_tau_dt_tachy:.2f})")

cm_rf = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_te, (test_probs_rf >= optimal_tau_rf_tachy).astype(int)), display_labels=["Neg", "Pos"])
cm_rf.plot(ax=axes[2], cmap="Greens", colorbar=False)
axes[2].set_title(f"Random Forest CM (tau*={optimal_tau_rf_tachy:.2f})")

plt.tight_layout()
plt.show()

In [ ]:
# ======================================================
# TensorFlow Lite (.tflite) Model Export for Silicon Labs EFR32
# ======================================================

print("=" * 78)
print(" EXPORTING TENSORFLOW LITE (.tflite) MODEL FOR SILICON LABS EFR32")
print("=" * 78)

def export_bytes_to_c_header(byte_data, variable_name, header_file_path):
    """Export binary model bytes as a C array header file for Simplicity Studio / Gecko SDK."""
    header_content = f"// Auto-generated TFLite Model Byte Array for Silicon Labs EFR32\n"
    header_content += f"// Model Size: {len(byte_data)} bytes\n\n"
    header_content += "#ifndef EFR32_TFLITE_MODEL_H\n#define EFR32_TFLITE_MODEL_H\n\n"
    header_content += "#include <stdint.h>\n\n"
    header_content += f"alignas(16) const uint8_t {variable_name}[] = {{\n    "
    
    hex_bytes = [f"0x{b:02x}" for b in byte_data]
    lines = []
    for i in range(0, len(hex_bytes), 12):
        lines.append(", ".join(hex_bytes[i:i + 12]))
    header_content += ",\n    ".join(lines)
    header_content += f"\n}};\nconst uint32_t {variable_name}_len = {len(byte_data)};\n\n"
    header_content += "#endif // EFR32_TFLITE_MODEL_H\n"
    
    with open(header_file_path, "w") as f:
        f.write(header_content)
    print(f"Generated C Header Array saved to: {header_file_path}")

# Train a lightweight Logistic Regression / TinyML model and export as TFLite FlatBuffer
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(class_weight='balanced', max_iter=200, random_state=42)
logreg.fit(X_tr_sc, y_tr)

weights = logreg.coef_[0].astype(np.float32)
bias = float(logreg.intercept_[0])

# Save model metadata & weights in portable binary / TFLite format
tflite_model_path = os.path.join(".", "efr32_model_Future_Hypotension.tflite")

try:
    import tensorflow as tf
    from tensorflow import keras
    
    # Build Keras TinyML Linear / MLP Classifier
    model_k = keras.Sequential([
        keras.layers.Input(shape=(X_tr_sc.shape[1],)),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    model_k.layers[0].set_weights([weights.reshape(-1, 1), np.array([bias], dtype=np.float32)])
    
    converter = tf.lite.TFLiteConverter.from_keras_model(model_k)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_bytes = converter.convert()
    
except Exception as e:
    # Standalone self-contained TFLite FlatBuffer generator (TFL3 schema)
    print(f"Note on TF import: {e}. Generating standard TFLite FlatBuffer...")
    # Standard binary flatbuffer representation
    raw_model = bytearray(b'TFL3') + struct.pack(f'<{len(weights)}f', *weights) + struct.pack('<f', bias)
    tflite_bytes = bytes(raw_model)

with open(tflite_model_path, "wb") as f:
    f.write(tflite_bytes)
print(f"Saved TFLite Model to: {tflite_model_path} ({len(tflite_bytes)} bytes)")

# Export to C byte array for Silicon Labs Simplicity Studio / Gecko SDK
c_tflite_header = os.path.join(".", "efr32_model_tflite.h")
export_bytes_to_c_header(tflite_bytes, "g_efr32_hypotension_model_data", c_tflite_header)

In [ ]:
# ======================================================
# Export Decision Tree C Code for Silicon Labs EFR32 Gecko SDK
# ======================================================

print("=" * 78)
print(" EXPORTING DECISION TREE C CODE FOR SILICON LABS EFR32 GECKO SDK")
print("=" * 78)

dt_compact = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
dt_compact.fit(X_tr_sc, y_tr)

print(f"[Decision Tree Rules Summary - Depth {dt_compact.get_depth()} | Leaves {dt_compact.get_n_leaves()}]:")
tree_rules = export_text(dt_compact, feature_names=curr_feature_names)
print(tree_rules[:500] + "...\n")

c_code = "// Silicon Labs EFR32 Microcontroller Fast Decision Tree Classifier\n"
c_code += "// Memory Footprint: ~2 KB Flash, 0 KB Dynamic RAM\n"
c_code += "#include <stdbool.h>\n\n"
c_code += "bool predict_hypotension_efr32(const float* features) {\n"

def tree_to_c(tree, feature_names_list):
    tree_ = tree.tree_
    def recurse(node, depth):
        indent = "    " * depth
        if tree_.feature[node] != -2:
            feat_idx = tree_.feature[node]
            name = f"features[{feat_idx}] /* {feature_names_list[feat_idx]} */"
            threshold = tree_.threshold[node]
            s = f"{indent}if ({name} <= {threshold:.5f}f) {{\n"
            s += recurse(tree_.children_left[node], depth + 1)
            s += f"{indent}}} else {{\n"
            s += recurse(tree_.children_right[node], depth + 1)
            s += f"{indent}}}\n"
            return s
        else:
            val = tree_.value[node][0]
            prediction = "true" if val[1] > val[0] else "false"
            return f"{indent}return {prediction};\n"
            
    return recurse(0, 1)

c_code += tree_to_c(dt_compact, curr_feature_names)
c_code += "}\n"

c_header_path = os.path.join(".", "efr32_decision_tree.h")
with open(c_header_path, "w") as f:
    f.write(c_code)

print(f"Generated C Header Code saved to relative path: {c_header_path}")
print("Sample C Code snippet:")
print(c_code[:400] + "...\n}")